In [3]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
from openrouter import OpenRouter
from pydantic import BaseModel, Field

# ─────────────────────────────────────────────
# PRETTY OUTPUT — MODULES LOADED
# ─────────────────────────────────────────────
print("\n" + "═" * 68)
print("\033[1;36m  📦  MODULES LOADED\033[0m")
print("\033[90m  Standard · Data · LLM · Agent libraries ready\033[0m")
print("═" * 68)

modules = [
    ("📚 Standard",  ["json", "os", "time", "pathlib", "typing"]),
    ("📊 Data",      ["pandas", "plotly.express"]),
    ("🔐 Config",    ["dotenv"]),
    ("💻 Notebook",  ["IPython.display"]),
    ("🦜 LangChain", ["langchain.agents", "langchain_core.tools", "langchain_openai"]),
    ("🤖 LLM APIs",  ["openai", "openrouter"]),
    ("✅ Validation",["pydantic"]),
]

for category, mods in modules:
    print(f"\n  \033[1;33m{category}\033[0m")
    for m in mods:
        print(f"     \033[1;32m✓\033[0m \033[90m{m}\033[0m")

total = sum(len(mods) for _, mods in modules)
print(f"\n  \033[1;34m📊 Total:\033[0m \033[1;32m{total}\033[0m modules imported successfully")

print("\n" + "═" * 68 + "\n")


════════════════════════════════════════════════════════════════════
  📦  MODULES LOADED
  Standard · Data · LLM · Agent libraries ready
════════════════════════════════════════════════════════════════════

  📚 Standard
     ✓ json
     ✓ os
     ✓ time
     ✓ pathlib
     ✓ typing

  📊 Data
     ✓ pandas
     ✓ plotly.express

  🔐 Config
     ✓ dotenv

  💻 Notebook
     ✓ IPython.display

  🦜 LangChain
     ✓ langchain.agents
     ✓ langchain_core.tools
     ✓ langchain_openai

  🤖 LLM APIs
     ✓ openai
     ✓ openrouter

  ✅ Validation
     ✓ pydantic

  📊 Total: 15 modules imported successfully

════════════════════════════════════════════════════════════════════



## OpenRouter

In [5]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# ─────────────────────────────────────────────
# PRETTY OUTPUT — OPENROUTER CLIENT
# ─────────────────────────────────────────────
print("\n" + "═" * 68)
print("\033[1;36m  🌐  OPENROUTER CLIENT\033[0m")
print("═" * 68)

if openrouter_api_key:
    # Show masked key for security
    masked_key = f"{openrouter_api_key[:8]}...{openrouter_api_key[-4:]}" if len(openrouter_api_key) > 12 else "***"
    print(f"\n  \033[1;34m▸ API Key\033[0m      → \033[1;32m✅ FOUND\033[0m  \033[90m({masked_key})\033[0m")
    print(f"  \033[1;34m▸ Base URL\033[0m     → \033[90m{OPENROUTER_BASE_URL}\033[0m")
    print(f"  \033[1;34m▸ Timeout\033[0m      → \033[1;37m120\033[0m sec")
    print(f"  \033[1;34m▸ Max retries\033[0m  → \033[1;37m5\033[0m")
    status_line = "\033[1;32m✅ Client initialized successfully\033[0m"
else:
    print(f"\n  \033[1;34m▸ API Key\033[0m      → \033[1;31m❌ MISSING\033[0m")
    print(f"  \033[1;34m▸ Base URL\033[0m     → \033[90m{OPENROUTER_BASE_URL}\033[0m")
    print(f"\n  \033[1;33m⚠️  No OpenRouter API key found.\033[0m")
    print(f"     \033[90mAdd OPENROUTER_API_KEY to the .env file\033[0m")
    status_line = "\033[1;31m❌ Client not initialized (missing key)\033[0m"

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

print(f"\n  {status_line}")
print("\n" + "═" * 68 + "\n")


════════════════════════════════════════════════════════════════════
  🌐  OPENROUTER CLIENT
════════════════════════════════════════════════════════════════════

  ▸ API Key      → ✅ FOUND  (sk-or-v1...67a3)
  ▸ Base URL     → https://openrouter.ai/api/v1
  ▸ Timeout      → 120 sec
  ▸ Max retries  → 5

  ✅ Client initialized successfully

════════════════════════════════════════════════════════════════════



## OpenRouter

In [8]:
messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in the style of a pirate."},
    {"role": "user", "content": "What is the meaning of the universe?"}
]

model_id = "openrouter/free"

print("\n" + "═" * 68)
print("\033[1;36m  🏴‍☠️  OPENROUTER — FREE MODELS ROUTER\033[0m")
print("═" * 68)
print(f"\n  \033[1;34m▸ Model\033[0m        → \033[1;37m{model_id}\033[0m  \033[90m(auto-selects free model)\033[0m")
print(f"  \033[1;34m▸ System\033[0m       → \033[90mPirate style\033[0m")
print(f"  \033[1;34m▸ Question\033[0m     → \033[90mWhat is the meaning of the universe?\033[0m")

try:
    response = openrouter.chat.completions.create(
        model=model_id,
        messages=messages
    )
    
    answer = response.choices[0].message.content
    # Which model has been selected
    actual_model = response.model
    
    print(f"\n  \033[1;32m✅ Response received\033[0m")
    print(f"  \033[1;34m▸ Routed to\033[0m    → \033[1;33m{actual_model}\033[0m")
    print(f"\n  \033[1;33m🏴‍☠️  Pirate says:\033[0m")
    print(f"     \033[90m{answer}\033[0m")
    
except Exception as e:
    print(f"\n  \033[1;31m❌ Error:\033[0m \033[90m{e}\033[0m")

print("\n" + "═" * 68 + "\n")


════════════════════════════════════════════════════════════════════
  🏴‍☠️  OPENROUTER — FREE MODELS ROUTER
════════════════════════════════════════════════════════════════════

  ▸ Model        → openrouter/free  (auto-selects free model)
  ▸ System       → Pirate style
  ▸ Question     → What is the meaning of the universe?

  ✅ Response received
  ▸ Routed to    → nvidia/nemotron-3.5-super-vl:free

  🏴‍☠️  Pirate says:
     Ahoy there, ye curious landlubber! 

Shiver me timbers, that be a question that could make a Kraken scratch its head! Ye be askin' about the meanin' of the whole wide universe? 

Well, listen close: I reckon the universe be like a endless, glitterin' ocean, and we're all just sailors searchin' for the X that marks the spot. Some say the meanin' be gold and jewels; others say it be the thrill of the chase, the salt in the air, and the wind in yer sails.

Perhaps the true treasure ain't some chest buried on a distant shore, but the voyage itself—the storms we wea

## Data Loading

In [11]:
DATA = Path("../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels


# ─────────────────────────────────────────────
# OUTPUT — DATASET SUMMARY
# ─────────────────────────────────────────────
print("\n" + "═" * 68)
print("\033[1;36m  📂  CLAIMS DATASET\033[0m")
print("\033[90m  JSON seed files · claims + handler decisions\033[0m")
print("═" * 68)

print(f"\n  \033[1;33m⚙️   Configuration:\033[0m")
print(f"     \033[90m▸ Data dir\033[0m        → \033[1;37m{DATA}\033[0m")
print(f"     \033[90m▸ Main files\033[0m      → \033[37mclaims.json\033[0m + \033[37mevals/decisions.json\033[0m")
print(f"     \033[90m▸ Held-out files\033[0m  → \033[37mevals/held_out_claims.json\033[0m + \033[37mevals/held_out_decisions.json\033[0m")

print(f"\n  \033[1;33m📋  Dataset variants:\033[0m")
print(f"     \033[1;34m[1]\033[0m \033[1;37mget_sample_data()\033[0m                  → full queue")
print(f"     \033[1;34m[2]\033[0m \033[1;37mget_sample_data(tiny=True)\033[0m         → 5-claim subset")
print(f"     \033[1;34m[3]\033[0m \033[1;37mget_sample_data(held_out=True)\033[0m     → 25 held-out claims")
print(f"     \033[1;34m[4]\033[0m \033[1;37mget_sample_data(tiny=True, held_out=True)\033[0m → held-out subset")

# ─────────────────────────────────────────────
# QUICK LOAD CHECK
# ─────────────────────────────────────────────
try:
    _claims, _labels = get_sample_data()
    claims_ok = True
    n_claims = len(_claims)
    n_labels = len(_labels)
except Exception as e:
    claims_ok = False
    n_claims, n_labels = 0, 0
    err = str(e)

print(f"\n  \033[1;33m🔍  Load check (default queue):\033[0m")

if claims_ok:
    print(f"     \033[1;32m✅ Loaded successfully\033[0m")
    print(f"     \033[90m▸ Claims\033[0m          → \033[1;37m{n_claims}\033[0m")
    print(f"     \033[90m▸ Labeled decisions\033[0m → \033[1;37m{n_labels}\033[0m")
    matched = sum(1 for c in _claims if c["claim_id"] in _labels)
    print(f"     \033[90m▸ Matched IDs\033[0m     → \033[1;32m{matched}\033[0m / {n_claims}")
else:
    print(f"     \033[1;31m❌ Load failed:\033[0m \033[90m{err}\033[0m")
    print(f"     \033[90mCheck that {DATA} contains the expected JSON files.\033[0m")

print("\n" + "═" * 68 + "\n")


════════════════════════════════════════════════════════════════════
  📂  CLAIMS DATASET
  JSON seed files · claims + handler decisions
════════════════════════════════════════════════════════════════════

  ⚙️   Configuration:
     ▸ Data dir        → ..\data
     ▸ Main files      → claims.json + evals/decisions.json
     ▸ Held-out files  → evals/held_out_claims.json + evals/held_out_decisions.json

  📋  Dataset variants:
     [1] get_sample_data()                  → full queue
     [2] get_sample_data(tiny=True)         → 5-claim subset
     [3] get_sample_data(held_out=True)     → 25 held-out claims
     [4] get_sample_data(tiny=True, held_out=True) → held-out subset

  🔍  Load check (default queue):
     ✅ Loaded successfully
     ▸ Claims          → 25
     ▸ Labeled decisions → 25
     ▸ Matched IDs     → 25 / 25

════════════════════════════════════════════════════════════════════



In [14]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

load_dotenv(override=True)

## LLM
llm = ChatOpenAI(
    model=os.getenv("EVREN_MODEL"),          # deepseek-v4-flash
    api_key=os.getenv("EVREN_API_KEY"),     
    base_url=os.getenv("EVREN_BASE_URL"),
    temperature=0    # https://evren-llmapi.ssyz.org.tr/v1
)   

In [15]:
TINY = False

claims, labels = get_sample_data(tiny=TINY)
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

# ─────────────────────────────────────────────
# PRETTY OUTPUT — DATASET LOADED
# ─────────────────────────────────────────────
print("\n" + "═" * 68)
print("\033[1;36m  📊  DATASET LOADED\033[0m")
print("═" * 68)

print(f"\n  \033[1;33m⚙️   Mode:\033[0m")
tiny_display = "\033[1;32mON\033[0m  \033[90m(tiny subset)\033[0m" if TINY else "\033[90mOFF\033[0m  \033[90m(full dataset)\033[0m"
print(f"     \033[90m▸ TINY\033[0m            → {tiny_display}")

# ─────────────────────────────────────────────
# DATA SUMMARY
# ─────────────────────────────────────────────
print(f"\n  \033[1;33m📦  Loaded artifacts:\033[0m")
print(f"     \033[90m▸ Claims\033[0m          → \033[1;37m{len(claims):>4}\033[0m")
print(f"     \033[90m▸ Labels\033[0m          → \033[1;37m{len(labels):>4}\033[0m")
print(f"     \033[90m▸ Policies\033[0m        → \033[1;37m{len(policies):>4}\033[0m")

# ─────────────────────────────────────────────
# CONSISTENCY CHECK
# ─────────────────────────────────────────────
matched = sum(1 for c in claims if c["claim_id"] in labels)
coverage = matched / len(claims) * 100 if claims else 0

if matched == len(claims):
    coverage_display = "\033[1;32m✅ 100% coverage\033[0m"
else:
    coverage_display = f"\033[1;31m⚠️  {coverage:.0f}% coverage\033[0m"

print(f"\n  \033[1;33m🔍  Label coverage:\033[0m")
print(f"     \033[90m▸ Matched claim IDs\033[0m → \033[1;37m{matched}\033[0m / {len(claims)}  {coverage_display}")

# ─────────────────────────────────────────────
# POLICY WORDING
# ─────────────────────────────────────────────
wording_words = len(policy_wording.split())
wording_chars = len(policy_wording)

print(f"\n  \033[1;33m📜  Policy wording:\033[0m")
print(f"     \033[90m▸ Words\033[0m           → \033[1;37m{wording_words:,}\033[0m")
print(f"     \033[90m▸ Characters\033[0m      → \033[90m{wording_chars:,}\033[0m")
print(f"     \033[90m▸ Approx. tokens\033[0m  → \033[90m~{wording_words * 4 // 3:,}\033[0m  \033[90m(4 chars ≈ 1 token)\033[0m")

print("\n" + "═" * 68 + "\n")


════════════════════════════════════════════════════════════════════
  📊  DATASET LOADED
════════════════════════════════════════════════════════════════════

  ⚙️   Mode:
     ▸ TINY            → OFF  (full dataset)

  📦  Loaded artifacts:
     ▸ Claims          →   25
     ▸ Labels          →   25
     ▸ Policies        →   50

  🔍  Label coverage:
     ▸ Matched claim IDs → 25 / 25  ✅ 100% coverage

  📜  Policy wording:
     ▸ Words           → 1,369
     ▸ Characters      → 7,712
     ▸ Approx. tokens  → ~1,825  (4 chars ≈ 1 token)

════════════════════════════════════════════════════════════════════



In [16]:
claims[0]

{'claim_id': 'CLM-1001',
 'policy_id': 'POL-2001',
 'customer_id': 'CUS-3001',
 'claim_type': 'trip_delay',
 'submitted_on': '2026-06-24',
 'incident_date': '2026-06-18',
 'incident_location': 'Rome, Italy',
 'summary': 'Flight home canceled by the airline, rebooked the next morning',
 'story': 'My flight home from Rome on June 18 was canceled by Meridian Air about two hours before departure because of a crew problem. The next seat they could give me was at 6:55 the following morning. I booked a hotel near the airport, had dinner and breakfast there, and took taxis between the airport and the hotel. I am claiming the hotel, the meals and the taxis, and all of the receipts are attached.',
 'items': [{'description': 'Airport hotel, one night',
   'category': 'accommodation',
   'amount': 245.0},
  {'description': 'Dinner and breakfast', 'category': 'meals', 'amount': 95.0},
  {'description': 'Taxis between the airport and the hotel',
   'category': 'transport',
   'amount': 80.0}],
 'amo

In [17]:
labels[claims[0]["claim_id"]]

{'claim_id': 'CLM-1001',
 'decision': 'approve',
 'payout': 420.0,
 'over_approval_limit': False,
 'handler_id': 'HND-02',
 'decided_on': '2026-06-26',
 'reason': 'The carrier canceled the flight for a crew problem and the delay was 19 hours, well past the 6 hour waiting period on Premium. The hotel, meals and taxis are reasonable and all have receipts. The total is inside the $800 maximum.',
 'reference': 'Policy wording, section 3',
 'calculation': [{'step': 'Eligible expenses with receipts', 'amount': 420.0}],
 'tests': 'A routine trip delay with complete documents',
 'difficulty': 'routine',
 'failure_class': '',
 'needs': [],
 'tone': 'plain',
 'tiny': False}

## Exploring the Data

In [18]:
rows = []
for claim in claims:
    label = labels[claim["claim_id"]]
    rows.append({
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "claim_type": claim["claim_type"],
        "amount_claimed": claim["amount_claimed"],
        "difficulty": label["difficulty"],
        "decision": label["decision"],
        "payout": label["payout"],
    })

data = pd.DataFrame(rows)

# ─────────────────────────────────────────────
# PRETTY OUTPUT — DATAFRAME SUMMARY
# ─────────────────────────────────────────────
print("\n" + "═" * 72)
print("\033[1;36m  📋  CLAIMS DATAFRAME\033[0m")
print("═" * 72)

print(f"\n  \033[1;33m📐  Shape:\033[0m")
print(f"     \033[90m▸ Rows\033[0m           → \033[1;37m{len(data):>4}\033[0m  \033[90m(claims)\033[0m")
print(f"     \033[90m▸ Columns\033[0m        → \033[1;37m{len(data.columns):>4}\033[0m")

print(f"\n  \033[1;33m🏷️   Columns:\033[0m")
for col in data.columns:
    dtype = str(data[col].dtype)
    print(f"     \033[90m▸\033[0m \033[1;37m{col:<18}\033[0m \033[90m{dtype}\033[0m")

# ─────────────────────────────────────────────
# QUICK STATS
# ─────────────────────────────────────────────
if "decision" in data.columns:
    print(f"\n  \033[1;33m📊  Decision distribution:\033[0m")
    for decision, count in data["decision"].value_counts().items():
        pct = count / len(data) * 100
        bar_len = int(pct / 3)
        bar = "█" * bar_len + "\033[90m" + "░" * (33 - bar_len) + "\033[0m"
        print(f"     \033[1;34m▸\033[0m \033[1;37m{decision:<14}\033[0m "
              f"\033[1;32m{count:>3}\033[0m ({pct:>5.1f}%)  {bar}")

if "difficulty" in data.columns:
    print(f"\n  \033[1;33m🎯  Difficulty distribution:\033[0m")
    for diff, count in data["difficulty"].value_counts().items():
        pct = count / len(data) * 100
        bar_len = int(pct / 3)
        bar = "█" * bar_len + "\033[90m" + "░" * (33 - bar_len) + "\033[0m"
        print(f"     \033[1;34m▸\033[0m \033[1;37m{diff:<14}\033[0m "
              f"\033[1;32m{count:>3}\033[0m ({pct:>5.1f}%)  {bar}")

# ─────────────────────────────────────────────
# NUMERIC SUMMARY
# ─────────────────────────────────────────────
if "amount_claimed" in data.columns and "payout" in data.columns:
    print(f"\n  \033[1;33m💰  Financial summary:\033[0m")
    total_claimed = data["amount_claimed"].sum()
    total_paid = data["payout"].sum()
    print(f"     \033[90m▸ Total claimed\033[0m   → \033[1;37m${total_claimed:>12,.2f}\033[0m")
    print(f"     \033[90m▸ Total paid\033[0m      → \033[1;32m${total_paid:>12,.2f}\033[0m")
    print(f"     \033[90m▸ Payout ratio\033[0m    → \033[1;33m{(total_paid / total_claimed * 100 if total_claimed else 0):>11.1f}%\033[0m")
    print(f"     \033[90m▸ Avg claim\033[0m       → \033[90m${data['amount_claimed'].mean():>12,.2f}\033[0m")

# ─────────────────────────────────────────────
# PREVIEW
# ─────────────────────────────────────────────
print(f"\n  \033[1;33m👀  Preview (first 3 rows):\033[0m")
print("  " + "─" * 68)

preview = data.head(3)
for idx, row in preview.iterrows():
    print(f"\n  \033[1;34m[{idx}]\033[0m \033[90m{row['claim_id']}\033[0m")
    print(f"     \033[90m▸ Summary\033[0m     → {row['summary'][:60]}{'...' if len(row['summary']) > 60 else ''}")
    print(f"     \033[90m▸ Type\033[0m        → \033[1;37m{row['claim_type']}\033[0m")
    print(f"     \033[90m▸ Claimed\033[0m     → \033[37m${row['amount_claimed']:,.2f}\033[0m")
    print(f"     \033[90m▸ Difficulty\033[0m  → \033[1;33m{row['difficulty']}\033[0m")
    print(f"     \033[90m▸ Decision\033[0m    → \033[1;32m{row['decision']}\033[0m")
    print(f"     \033[90m▸ Payout\033[0m      → \033[1;32m${row['payout']:,.2f}\033[0m")

print("\n  " + "─" * 68)
print(f"  \033[90m💡 {len(data)} total rows — use \033[1;37mdata.head()\033[90m, \033[1;37mdata.info()\033[90m, \033[1;37mdata.describe()\033[90m for more.\033[0m")

print("\n" + "═" * 72 + "\n")


════════════════════════════════════════════════════════════════════════
  📋  CLAIMS DATAFRAME
════════════════════════════════════════════════════════════════════════

  📐  Shape:
     ▸ Rows           →   25  (claims)
     ▸ Columns        →    7

  🏷️   Columns:
     ▸ claim_id           str
     ▸ summary            str
     ▸ claim_type         str
     ▸ amount_claimed     float64
     ▸ difficulty         str
     ▸ decision           str
     ▸ payout             float64

  📊  Decision distribution:
     ▸ approve         14 ( 56.0%)  ██████████████████░░░░░░░░░░░░░░░
     ▸ refer            6 ( 24.0%)  ████████░░░░░░░░░░░░░░░░░░░░░░░░░
     ▸ decline          5 ( 20.0%)  ██████░░░░░░░░░░░░░░░░░░░░░░░░░░░

  🎯  Difficulty distribution:
     ▸ routine         15 ( 60.0%)  ████████████████████░░░░░░░░░░░░░
     ▸ hard             5 ( 20.0%)  ██████░░░░░░░░░░░░░░░░░░░░░░░░░░░
     ▸ manual           5 ( 20.0%)  ██████░░░░░░░░░░░░░░░░░░░░░░░░░░░

  💰  Financial summary:
     ▸ Tot

In [19]:
rows = []
for claim in claims:
    label = labels[claim["claim_id"]]
    rows.append({
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "claim_type": claim["claim_type"],
        "amount_claimed": claim["amount_claimed"],
        "difficulty": label["difficulty"],
        "decision": label["decision"],
        "payout": label["payout"],
    })

data = pd.DataFrame(rows)

# ─────────────────────────────────────────────
# PRETTY OUTPUT — FIRST 20 ROWS
# ─────────────────────────────────────────────
N = 20
preview = data.head(N)

print("\n" + "═" * 108)
print("\033[1;36m  📋  CLAIMS DATAFRAME\033[0m")
print(f"\033[90m  Showing first {len(preview)} of {len(data)} rows · {len(data.columns)} columns\033[0m")
print("═" * 108)

# ─────────────────────────────────────────────
# TABLE HEADER
# ─────────────────────────────────────────────
print(f"\n  \033[1;33m{'#':>3}  {'CLAIM ID':<12} {'TYPE':<18} {'CLAIMED':>11} "
      f"{'DIFFICULTY':<11} {'DECISION':<11} {'PAYOUT':>11}  SUMMARY\033[0m")
print("  " + "─" * 104)

# ─────────────────────────────────────────────
# TABLE ROWS
# ─────────────────────────────────────────────
for idx, row in preview.iterrows():
    # Color the decision
    decision = row["decision"]
    if decision == "approve":
        decision_display = f"\033[1;32m{decision:<11}\033[0m"
    elif decision == "deny":
        decision_display = f"\033[1;31m{decision:<11}\033[0m"
    elif decision == "escalate":
        decision_display = f"\033[1;33m{decision:<11}\033[0m"
    else:
        decision_display = f"\033[1;37m{decision:<11}\033[0m"

    # Color the difficulty
    difficulty = row["difficulty"]
    if difficulty == "easy":
        difficulty_display = f"\033[1;32m{difficulty:<11}\033[0m"
    elif difficulty == "medium":
        difficulty_display = f"\033[1;33m{difficulty:<11}\033[0m"
    elif difficulty == "hard":
        difficulty_display = f"\033[1;31m{difficulty:<11}\033[0m"
    else:
        difficulty_display = f"\033[1;37m{difficulty:<11}\033[0m"

    # Shorten summary to 28 characters
    summary = row["summary"]
    summary_short = summary[:28] + "…" if len(summary) > 28 else summary

    print(f"  \033[90m{idx:>3}\033[0m  \033[1;37m{row['claim_id']:<12}\033[0m "
          f"\033[90m{row['claim_type']:<18}\033[0m "
          f"\033[37m${row['amount_claimed']:>9,.2f}\033[0m "
          f"{difficulty_display} {decision_display} "
          f"\033[1;32m${row['payout']:>9,.2f}\033[0m  "
          f"\033[90m{summary_short}\033[0m")

print("  " + "─" * 104)

# ─────────────────────────────────────────────
# FOOTER
# ─────────────────────────────────────────────
print(f"\n  \033[1;34m📊  Total:\033[0m \033[1;37m{len(data)}\033[0m rows  ·  "
      f"\033[90m{len(data) - len(preview)} more not shown (use \033[1;37mdata.head(50)\033[90m or \033[1;37mdata.iloc[a:b]\033[90m)\033[0m")

print("\n" + "═" * 108 + "\n")


════════════════════════════════════════════════════════════════════════════════════════════════════════════
  📋  CLAIMS DATAFRAME
  Showing first 20 of 25 rows · 7 columns
════════════════════════════════════════════════════════════════════════════════════════════════════════════

    #  CLAIM ID     TYPE                   CLAIMED DIFFICULTY  DECISION         PAYOUT  SUMMARY
  ────────────────────────────────────────────────────────────────────────────────────────────────────────
    0  CLM-1001     trip_delay         $   420.00 routine     approve     $   420.00  Flight home canceled by the …
    1  CLM-1002     baggage            $ 1,150.00 routine     approve     $   580.00  Suitcase lost on a connectin…
    2  CLM-1003     emergency_medical  $ 3,800.00 routine     approve     $ 3,750.00  Hospital visit abroad after …
    3  CLM-1004     baggage_delay      $   186.00 routine     approve     $   186.00  Checked bag arrived 31 hours…
    4  CLM-1005     passport           $   243.00

In [20]:
home_alone = [claim for claim in claims if claim["claim_id"] == "CLM-1015"][0]
decision = labels["CLM-1015"]

display(Markdown(f"""**{home_alone['summary']}**

{home_alone['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

**Flew home from Paris on arrival after leaving a child at home**

We flew from Chicago to Paris on June 20 as a family of six. Somewhere over the Atlantic I realized that our youngest son, Kevin, who is eight, was not on the plane. We had overslept, we left for the airport in a rush and the head count in the van was wrong. I took the first available flight home the moment we landed in Paris. Kevin was fine. In his words, he had handled it. I am claiming my one-way fare home and my share of the prepaid hotel nights in Paris that I did not use.

**Handler:** decline $0.00. Trip Interruption pays only for the reasons listed in the wording. Returning home because a child was left behind is not one of them, and nobody was sick or injured.


In [21]:
queue, queue_labels = get_sample_data()
hangover = [claim for claim in queue if claim["claim_id"] == "CLM-1010"][0]
decision = queue_labels["CLM-1010"]

display(Markdown(f"""**{hangover['summary']}**

{hangover['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

**Emergency dental treatment in Las Vegas for a missing front tooth**

I was in Las Vegas from June 19 for a friend's bachelor party. On the morning of June 21 I woke up on the floor of our suite missing an upper front tooth. I have no memory of the night before, and neither do my friends. I am a dentist, and I have to say that the extraction is consistent with my own technique. I went to an emergency dental clinic that morning for an exam, an X-ray and a temporary bridge, and their bill and notes are attached. There was also a tiger in the bathroom of the suite, which I mention only for completeness. I am not claiming for anything connected with the tiger.

**Handler:** decline $0.00. The treatment was needed, and the bill and the notes are in order. The general exclusions rule out losses while the customer is impaired by alcohol or drugs. The clinic's notes record heavy drinking through the evening, no memory of the night and an extraction that the customer believes he carried out himself during it, so the injury happened while he was impaired and no benefit pays.


In [24]:
fig = px.histogram(data, x="claim_type", color="decision", title="Claims by type")
fig.show(renderer = "browser")